# Nanochat RL And DPO Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and runs Kaggle-friendly RL and DPO training passes.

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` expected)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes new checkpoints under `chatrl_checkpoints/d8_kaggle` and `chatdpo_checkpoints/d8_kaggle`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/yshuaiqin/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Nanochat cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies


In [2]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'requests>=2.32.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
    'rustbpe>=0.1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.4 MB/s eta 0:00:00
Dependencies installed


## Configure Runtime


In [3]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'

env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'

env['OMP_NUM_THREADS'] = '1'
env['PYTHONUNBUFFERED'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
env['NCCL_DEBUG'] = 'WARN'

os.environ.update(env)

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Validate Repo And SFT Cache


In [4]:
assert (WORK_REPO / 'scripts' / 'chat_rl.py').exists(), 'Missing scripts/chat_rl.py'
assert (WORK_REPO / 'scripts' / 'chat_dpo.py').exists(), 'Missing scripts/chat_dpo.py'
assert (WORK_REPO / 'scripts' / 'chat_post_eval.py').exists(), 'Missing scripts/chat_post_eval.py'
assert (WORK_REPO / 'scripts' / 'chat_web.py').exists(), 'Missing scripts/chat_web.py'

sft_dir = WORK_CACHE / 'chatsft_checkpoints' / 'd8_kaggle'
tokenizer_dir = WORK_CACHE / 'tokenizer'
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
assert tokenizer_dir.exists(), f'Missing tokenizer dir: {tokenizer_dir}'
assert sorted(sft_dir.glob('model_*.pt')), f'No model_*.pt files found in {sft_dir}'
assert sorted(sft_dir.glob('meta_*.json')), f'No meta_*.json files found in {sft_dir}'

subprocess.check_call(
    [
        sys.executable, '-m', 'py_compile',
        'scripts/chat_rl.py',
        'scripts/chat_dpo.py',
        'scripts/chat_post_eval.py',
        'scripts/chat_web.py',
    ],
    cwd=WORK_REPO,
    env=env,
)

print('Setup complete')
print('SFT checkpoints:', [p.name for p in sorted(sft_dir.glob('model_*.pt'))[-5:]])
print('Tokenizer files:', sorted(p.name for p in tokenizer_dir.iterdir()))


Setup complete
SFT checkpoints: ['model_004999.pt']
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## RL Run


In [5]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
MODEL_TAG = 'd8_kaggle'
OVERWRITE_RL_CHECKPOINTS = True

if OVERWRITE_RL_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatrl_checkpoints' / MODEL_TAG, ignore_errors=True)

rl_args = [
    '-m', 'scripts.chat_rl',
    '--',
    '--run=dummy',
    f'--model-tag={MODEL_TAG}',

    '--max-steps=60',
    '--device-batch-size=4',
    '--examples-per-step=4',
    '--num-samples=8',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=50',

    '--eval-every=20',
    '--eval-examples=50',
    '--save-every=20',

    '--embedding-lr=0.03',
    '--unembedding-lr=0.0008',
    '--matrix-lr=0.002',
    '--init-lr-frac=0.05',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + rl_args
else:
    cmd = [sys.executable] + rl_args

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_rl -- --run=dummy --model-tag=d8_kaggle --max-steps=60 --device-batch-size=4 --examples-per-step=4 --num-samples=8 --max-new-tokens=256 --temperature=1.0 --top-k=50 --eval-every=20 --eval-examples=50 --save-every=20 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.05


[W614 04:01:52.756426810 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 04:02:04,977 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:02:04,977 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:02:04,981 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 04:02:04,981 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 04:02:06.139860091 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 04:02:06.141242969 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

2026-06-14 04:12:07,005 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/model_000040.pt
2026-06-14 04:12:07,006 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/meta_000040.json


✅ Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle
Step 41/60 | Example step 0 | Pass 0 | loss: -0.000000 | Average reward: 0.0
Step 41/60 | Example step 0 | Pass 1 | loss: -0.000000 | Average reward: 0.0
Step 41/60 | Example step 1 | Pass 0 | loss: -0.000000 | Average reward: 0.0
Step 41/60 | Example step 1 | Pass 1 | loss: -0.000000 | Average reward: 0.0
Step 41/60 | Average reward: 0.0 | Average sequence length: 211.81
Step 42/60 | Example step 0 | Pass 0 | loss: -0.000000 | Average reward: 0.0
Step 42/60 | Example step 0 | Pass 1 | loss: -0.000000 | Average reward: 0.0
Step 42/60 | Example step 1 | Pass 0 | loss: -0.000000 | Average reward: 0.0
Step 42/60 | Example step 1 | Pass 1 | loss: -0.000000 | Average reward: 0.0
Step 42/60 | Average reward: 0.0 | Average sequence length: 237.53
Step 43/60 | Example step 0 | Pass 0 | loss: -0.000000 | Average reward: 0.0
Step 43/60 | Example step 0 | Pass 1 | loss: -0.000000 | Average reward: 0.0
Step 43/

2026-06-14 04:15:10,666 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/model_000059.pt
2026-06-14 04:15:10,666 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/meta_000059.json


✅ Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.chat_rl', '--', '--run=dummy', '--model-tag=d8_kaggle', '--max-steps=60', '--device-batch-size=4', '--examples-per-step=4', '--num-samples=8', '--max-new-tokens=256', '--temperature=1.0', '--top-k=50', '--eval-every=20', '--eval-examples=50', '--save-every=20', '--embedding-lr=0.03', '--unembedding-lr=0.0008', '--matrix-lr=0.002', '--init-lr-frac=0.05'], returncode=0)

## DPO Run


In [6]:
MODEL_TAG = 'd8_kaggle'
OVERWRITE_DPO_CHECKPOINTS = True

if OVERWRITE_DPO_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatdpo_checkpoints' / MODEL_TAG, ignore_errors=True)

sft_dir = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
print('Latest SFT checkpoint:', sorted(sft_dir.glob('model_*.pt'))[-1])

dpo_args = [
    '-m', 'scripts.chat_dpo',
    '--',
    '--run=dummy',

    '--model-source=sft',
    f'--model-tag={MODEL_TAG}',

    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',

    '--max-train-examples=2048',
    '--max-val-examples=256',
    '--batch-size=2',
    '--num-epochs=1',
    '--max-steps=300',

    '--beta=0.1',
    '--label-smoothing=0.0',

    '--embedding-lr=0.003',
    '--unembedding-lr=0.00008',
    '--matrix-lr=0.0002',
    '--init-lr-frac=0.2',

    '--eval-every=50',
    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + dpo_args
else:
    cmd = [sys.executable] + dpo_args

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


Latest SFT checkpoint: /kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle/model_004999.pt
Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_dpo -- --run=dummy --model-source=sft --model-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --preference-source=gsm8k --max-train-examples=2048 --max-val-examples=256 --batch-size=2 --num-epochs=1 --max-steps=300 --beta=0.1 --label-smoothing=0.0 --embedding-lr=0.003 --unembedding-lr=0.00008 --matrix-lr=0.0002 --init-lr-frac=0.2 --eval-every=50 --save-every=100


[W614 04:15:16.644528019 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 04:15:21,522 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:15:21,523 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 04:15:21,555 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:15:21,556 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 04:15:22.643835787 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 04:15:22.655149836 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

2026-06-14 04:16:46,556 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle/model_000200.pt
2026-06-14 04:16:46,556 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle/meta_000200.json


Saved DPO checkpoint to /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle
Step 201 | epoch 0 | dpo_loss: 0.022799 | pref_acc: 1.0000 | lrm: 1.0000
Step 202 | epoch 0 | dpo_loss: 0.027734 | pref_acc: 1.0000 | lrm: 1.0000
Step 203 | epoch 0 | dpo_loss: 0.022233 | pref_acc: 1.0000 | lrm: 1.0000
Step 204 | epoch 0 | dpo_loss: 0.015560 | pref_acc: 1.0000 | lrm: 1.0000
Step 205 | epoch 0 | dpo_loss: 0.015368 | pref_acc: 1.0000 | lrm: 1.0000
Step 206 | epoch 0 | dpo_loss: 0.015416 | pref_acc: 1.0000 | lrm: 1.0000
Step 207 | epoch 0 | dpo_loss: 0.021613 | pref_acc: 1.0000 | lrm: 1.0000
Step 208 | epoch 0 | dpo_loss: 0.016785 | pref_acc: 1.0000 | lrm: 1.0000
Step 209 | epoch 0 | dpo_loss: 0.022986 | pref_acc: 1.0000 | lrm: 1.0000
Step 210 | epoch 0 | dpo_loss: 0.018232 | pref_acc: 1.0000 | lrm: 1.0000
Step 211 | epoch 0 | dpo_loss: 0.022043 | pref_acc: 1.0000 | lrm: 1.0000
Step 212 | epoch 0 | dpo_loss: 0.018082 | pref_acc: 1.0000 | lrm: 1.0000
Step 213 | epoch 0 | dpo_loss: 0.021474

2026-06-14 04:17:22,330 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle/model_000300.pt
2026-06-14 04:17:22,331 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle/meta_000300.json


Saved final DPO checkpoint to /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '-m', 'scripts.chat_dpo', '--', '--run=dummy', '--model-source=sft', '--model-tag=d8_kaggle', '--reference-source=sft', '--reference-tag=d8_kaggle', '--preference-source=gsm8k', '--max-train-examples=2048', '--max-val-examples=256', '--batch-size=2', '--num-epochs=1', '--max-steps=300', '--beta=0.1', '--label-smoothing=0.0', '--embedding-lr=0.003', '--unembedding-lr=0.00008', '--matrix-lr=0.0002', '--init-lr-frac=0.2', '--eval-every=50', '--save-every=100'], returncode=0)

## Inspect RL And DPO Checkpoints


In [7]:
for family in ['chatrl_checkpoints', 'chatdpo_checkpoints']:
    checkpoint_dir = WORK_CACHE / family / MODEL_TAG
    print()
    print(family, checkpoint_dir)
    print('Exists:', checkpoint_dir.exists())
    if checkpoint_dir.exists():
        print('Model checkpoints:', [p.name for p in sorted(checkpoint_dir.glob('model_*.pt'))])
        print('Metadata files:', [p.name for p in sorted(checkpoint_dir.glob('meta_*.json'))])

subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)



chatrl_checkpoints /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle
Exists: True
Model checkpoints: ['model_000020.pt', 'model_000040.pt', 'model_000059.pt']
Metadata files: ['meta_000020.json', 'meta_000040.json', 'meta_000059.json']

chatdpo_checkpoints /kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle
Exists: True
Model checkpoints: ['model_000100.pt', 'model_000200.pt', 'model_000300.pt']
Metadata files: ['meta_000100.json', 'meta_000200.json', 'meta_000300.json']
4.9G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Comparison Eval


In [8]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 50

if RUN_COMPARISON_EVAL:
    models = ['sft=sft:d8_kaggle']

    rl_dir = WORK_CACHE / 'chatrl_checkpoints'
    if (rl_dir / MODEL_TAG).exists():
        models.append(f'rl=@{rl_dir}:{MODEL_TAG}')

    dpo_dir = WORK_CACHE / 'chatdpo_checkpoints'
    dpo_model_dir = dpo_dir / MODEL_TAG
    if dpo_model_dir.exists():
        model_steps = sorted(
            int(p.stem.split('_')[-1])
            for p in dpo_model_dir.glob('model_*.pt')
        )
        for step in model_steps:
            models.append(f'dpo{step}=@{dpo_dir}:{MODEL_TAG}:{step}')

    post_eval_args = [
        '-m', 'scripts.chat_post_eval',
        '--',
    ]
    for model in models:
        post_eval_args.extend(['--models', model])
    post_eval_args.extend([
        '--max-problems', str(EVAL_MAX_PROBLEMS),
        '--batch-size', '2',
    ])

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print('Skipping comparison eval')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_post_eval -- --models sft=sft:d8_kaggle --models rl=@/kaggle/working/nanochat_cache/chatrl_checkpoints:d8_kaggle --models dpo100=@/kaggle/working/nanochat_cache/chatdpo_checkpoints:d8_kaggle:100 --models dpo200=@/kaggle/working/nanochat_cache/chatdpo_checkpoints:d8_kaggle:200 --models dpo300=@/kaggle/working/nanochat_cache/chatdpo_checkpoints:d8_kaggle:300 --max-problems 50 --batch-size 2


[W614 04:17:26.065096377 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 04:17:30,891 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:17:30,893 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 04:17:30,955 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 04:17:30,957 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 04:17:31.314606698 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 04:17:31.361994147 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Final: 7/50 (14.00%)
dpo300 | MMLU: 14.00%


2026-06-14 04:36:23,561 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 04:36:23,562 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 04:36:23,567 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 04:36:23,568 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 04:36:23,583 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 04:36:23,584 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
dpo300 | GSM8K: 0.00%


2026-06-14 04:38:12,291 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 04:38:12,293 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 04:38:12,296 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 04:38:12,299 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 04:38:12,311 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/openai_humaneval.py "HTTP/1.1 404 Not Found"
2026-06-14 04:38:12,318 - httpx - INFO

Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
dpo300 | HumanEval: 0.00%
Rank 1 | 0/25 (0.00%)
Rank 0 | 0/25 (0.00%)
Final: 0/50 (0.00%)
dpo300 | SpellingBee: 0.00%

Model  | ARC-Easy | ARC-Challenge | MMLU   | GSM8K | HumanEval | SpellingBee | Mean  
-------+----------+---------------+--------+-------+-----------+-------------+-------
sft    | 30.00%   | 30.00%        | 32.00% | 0.00% | 0.00%     | 96.00%      | 31.33%
rl     | 34.00%   | 26.00%        | 24.00% | 0.00% | 0.00%     | 92.00%      | 29.33%
dpo100 | 34.00%   | 30.00%        | 36.00% | 0.00% | 0.00%     | 0.00%       | 16.67%
dpo200 | 36.00%   | 24.00%        | 28.00% | 0.00% | 0.00%     | 0.00%       | 14.67%
dpo300 | 22.00%   | 18.00%        | 14.00% | 0.00% | 0.00%     | 0.00%       | 9.00% 

Ranking by mean score:
1. sft: 31.33%
2. rl: 29.33%
3. dpo100: 16.67%
4. dpo200: 14.67%
5. dpo300: 9.00%


## Inspect Saved Comparison Report


In [9]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print('Exists:', report_path.exists())
if report_path.exists():
    print(report_path.read_text())


/kaggle/working/nanochat_cache/report/chat-post-eval.md
Exists: True
## Chat Post Eval
timestamp: 2026-06-14 04:42:05

- tasks: ['ARC-Easy', 'ARC-Challenge', 'MMLU', 'GSM8K', 'HumanEval', 'SpellingBee']
- num_samples: 1
- temperature: 0.0000
- max_new_tokens: 512
- batch_size: 2
- max_problems: 50
- models: [{'label': 'sft', 'origin': 'sft', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 4999, 'resolved_meta_keys': ['model_config', 'step', 'user_config', 'val_bpb']}, {'label': 'rl', 'origin': '/kaggle/working/nanochat_cache/chatrl_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 59, 'resolved_meta_keys': ['model_config']}, {'label': 'dpo100', 'origin': '/kaggle/working/nanochat_cache/chatdpo_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 100, 'resolved_meta_

## Output Manifest


In [10]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG),
    'rl_checkpoint_dir': str(WORK_CACHE / 'chatrl_checkpoints' / MODEL_TAG),
    'dpo_checkpoint_dir': str(WORK_CACHE / 'chatdpo_checkpoints' / MODEL_TAG),
    'report': str(WORK_CACHE / 'report' / 'chat-post-eval.md'),
}
manifest_path = Path('/kaggle/working/nanochat_rl_dpo_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())

print('Main output files:')
for family in ['chatrl_checkpoints', 'chatdpo_checkpoints']:
    for path in sorted((WORK_CACHE / family / MODEL_TAG).glob('*')):
        print(path, path.stat().st_size, 'bytes')


/kaggle/working/nanochat_rl_dpo_manifest.json
{
  "model_tag": "d8_kaggle",
  "sft_checkpoint_dir": "/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle",
  "rl_checkpoint_dir": "/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle",
  "dpo_checkpoint_dir": "/kaggle/working/nanochat_cache/chatdpo_checkpoints/d8_kaggle",
  "report": "/kaggle/working/nanochat_cache/report/chat-post-eval.md"
}
Main output files:
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/meta_000020.json 178 bytes
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/meta_000040.json 178 bytes
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/meta_000059.json 178 bytes
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/model_000020.pt 503342527 bytes
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/model_000040.pt 503342527 bytes
/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle/model_000059.pt 503342527 bytes
/kaggle/working/nanochat_cache/cha

In [11]:
# Optional: save the RL/DPO checkpoint cache as a Kaggle Dataset.
import json

RL_DPO_MODEL_TAG = MODEL_TAG
RL_DPO_CACHE_DIR = Path('/kaggle/working/nanochat_d8_rl_dpo_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-rl-dpo-cache'
TITLE = 'nanochat d8 rl dpo cache'
VERSION_MESSAGE = f'Save {RL_DPO_MODEL_TAG} RL and DPO checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

rl_source_dir = WORK_CACHE / 'chatrl_checkpoints' / RL_DPO_MODEL_TAG
dpo_source_dir = WORK_CACHE / 'chatdpo_checkpoints' / RL_DPO_MODEL_TAG
tokenizer_source_dir = WORK_CACHE / 'tokenizer'

assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
assert tokenizer_source_dir.exists(), f'Missing tokenizer directory: {tokenizer_source_dir}'
assert rl_source_dir.exists(), f'Missing RL checkpoint directory: {rl_source_dir}'
assert dpo_source_dir.exists(), f'Missing DPO checkpoint directory: {dpo_source_dir}'
assert sorted(rl_source_dir.glob('model_*.pt')), f'No model_*.pt files found in {rl_source_dir}'
assert sorted(dpo_source_dir.glob('model_*.pt')), f'No model_*.pt files found in {dpo_source_dir}'

if RL_DPO_CACHE_DIR.exists():
    shutil.rmtree(RL_DPO_CACHE_DIR)
RL_DPO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(WORK_CACHE / 'chatrl_checkpoints', RL_DPO_CACHE_DIR / 'chatrl_checkpoints')
shutil.copytree(WORK_CACHE / 'chatdpo_checkpoints', RL_DPO_CACHE_DIR / 'chatdpo_checkpoints')
shutil.copytree(tokenizer_source_dir, RL_DPO_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = RL_DPO_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(RL_DPO_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(RL_DPO_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(RL_DPO_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
2.9G	/kaggle/working/nanochat_d8_rl_dpo_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_rl_dpo_cache --dir-mode tar
Starting upload for file chatdpo_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:11<00:00, 127MB/s]


Upload successful: chatdpo_checkpoints.tar (1GB)
Starting upload for file chatrl_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:14<00:00, 104MB/s]


Upload successful: chatrl_checkpoints.tar (1GB)
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 2.09MB/s]


Upload successful: tokenizer.tar (540KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-rl-dpo-cache
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-rl-dpo-cache


## Serve the RL/DPO Chat Model

Run this after `chatrl_checkpoints/d8_kaggle` or `chatdpo_checkpoints/d8_kaggle` exists. Set `SERVE_SOURCE` to `dpo` or `rl` before running the first cell.


In [ ]:
import time
import requests

SERVE_SOURCE = 'dpo'  # choose 'dpo' or 'rl'
SERVE_MODEL_TAG = MODEL_TAG
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'

checkpoint_family = {
    'dpo': 'chatdpo_checkpoints',
    'rl': 'chatrl_checkpoints',
}[SERVE_SOURCE]
CHECKPOINT_DIR = WORK_CACHE / checkpoint_family

model_dir = CHECKPOINT_DIR / SERVE_MODEL_TAG
assert model_dir.exists(), f'Missing {SERVE_SOURCE} checkpoint directory: {model_dir}'
assert sorted(model_dir.glob('model_*.pt')), f'No model_*.pt files found in {model_dir}'
assert sorted(model_dir.glob('meta_*.json')), f'No meta_*.json files found in {model_dir}'

try:
    if server.poll() is None:
        print('Stopping existing NanoChat server...')
        server.terminate()
        server.wait(timeout=10)
        print('Stopped existing server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing server cleanly:', exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print('Killed existing server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--model-tag', SERVE_MODEL_TAG,
    '--num-gpus', '1',
    '--port', str(SERVER_PORT),
]

print('Starting NanoChat server:')
print(' '.join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Server process started with PID {server.pid}.')

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f'NanoChat server exited early with code {server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat server is ready: {BASE_URL}')
else:
    print(f'NanoChat server is still loading. Wait a bit, then use: {BASE_URL}')

In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')